# Phenology Extraction

This recipe walks through evy's **phenology module** — tools for identifying the Start of Season (SOS), Middle of Season (MOS), and End of Season (EOS) from EVI time series. The quickstart demonstrates `calculate_phenology` as a one-liner; here we unpack the full pipeline.

evy's phenology approach is based on [TIMESAT](https://web.nateko.lu.se/timesat/timesat.asp) (Jönsson & Eklundh 2004): Savitzky-Golay smoothing followed by amplitude-threshold detection. See [Design Decisions](../../docs/design-decisions.md) for the methodological rationale.

**Known limitation:** The current implementation assumes a single growing season per year. Regions with double cropping (e.g., irrigated rice) will show only the dominant season.

In [ ]:
# papermill parameters
# Use string literals (not evy.ORIGINAL etc.) so this cell can run before evy is imported.
country = "RWA"
admin_level = 1
start_date = "2023-01-01"
end_date = "2023-12-31"
freq = "Original"  # evy.ORIGINAL
quick_mode = False


In [ ]:
if quick_mode:
    # daily is wasteful for a smoke test; monthly still drives the full pipeline.
    freq = "ME"  # evy.MONTHLY
    end_date = "2023-06-30"


In [ ]:
import evy
import numpy as np
import pandas as pd

## Step 1: Compute daily EVI

Phenology extraction works best with high-frequency data. We keep the original satellite composites (`freq="Original"`, about one image every 8 days from Terra and Aqua combined) so that the smoothing step has enough data points to resolve the seasonal curve.

In [ ]:
gdf = evy.get_boundaries(country, admin_level=admin_level)
df = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date=start_date,
    end_date=end_date,
    freq=freq,
    stats=["mean"],
)
print(f"{len(df)} rows, {df['shapeName'].nunique()} regions")
df.head()

## Step 2: High-level phenology — `calculate_phenology`

The simplest path. One call aggregates to monthly values, smooths them, and extracts SOS/MOS/EOS.

In [ ]:
phenology = evy.calculate_phenology(df, value_col="mean")
phenology

In [ ]:
evy.plot_seasonality(phenology)

## Step 3: Per-region phenology

Pass `group_col` to compute phenology per administrative unit. Regions with different climates or cropping calendars will show different season timing.

In [ ]:
phenology_by_region = evy.calculate_phenology(
    df, value_col="mean", group_col="shapeName"
)
phenology_by_region.head(12)

In [ ]:
evy.plot_seasonality_by_region(phenology_by_region, region_col="shapeName")

## Step 4: Under the hood — `preprocess_series` and `extract_phenology`

`calculate_phenology` is a convenience wrapper. Under the hood it calls:

1. **`preprocess_series`** — outlier removal, interpolation, Savitzky-Golay smoothing
2. **`extract_phenology`** — amplitude-threshold SOS/MOS/EOS detection

Let's run them manually on one region's monthly averages to see the intermediate steps.

In [ ]:
# Pick one region's monthly data
region = phenology_by_region["shapeName"].iloc[0]
region_data = phenology_by_region[phenology_by_region["shapeName"] == region]

# preprocess_series expects a pd.Series indexed by month
series = pd.Series(
    region_data["value"].values,
    index=region_data["month"].values,
)

smoothed = evy.preprocess_series(series)
print(f"Region: {region}")
print(f"Raw values:    {np.round(series.values, 3)}")
print(f"Smoothed:      {np.round(smoothed, 3)}")

In [ ]:
# Extract SOS, MOS, EOS from the smoothed series
metrics = evy.extract_phenology(smoothed, dates=series.index)
print(metrics)

## Step 5: Determine and filter the growing season

`get_growing_season` returns the start and end months. `filter_growing_season` subsets a DataFrame to that window.

In [ ]:
start_month, end_month = evy.get_growing_season(df, value_col="mean")
print(f"Growing season: month {start_month} to month {end_month}")

In [ ]:
df_growing = evy.filter_growing_season(
    df, start_month=start_month, end_month=end_month
)
print(f"Full dataset:    {len(df)} rows")
print(f"Growing season:  {len(df_growing)} rows")
df_growing.head()

## Summary

| Function | Purpose |
|----------|---------|
| `calculate_phenology` | One-call phenology extraction (aggregates, smooths, extracts) |
| `preprocess_series` | TIMESAT-style outlier removal + Savitzky-Golay smoothing |
| `extract_phenology` | Amplitude-threshold SOS/MOS/EOS detection on a smoothed series |
| `get_growing_season` | Returns (start_month, end_month) tuple |
| `filter_growing_season` | Subsets a DataFrame to the growing season window |
| `plot_seasonality` | Seasonality chart with raw + smoothed values and SOS/MOS/EOS markers |
| `plot_seasonality_by_region` | Faceted version of the above, one panel per region |

For the methodological background behind these choices, see [Design Decisions](../../docs/design-decisions.md).